

Proposed Structure:


&ensp; \[Manual VS Automatic Curation\]

&ensp;&ensp; \[Processing Stage \]

&ensp;&ensp; \[File Format ( PDB VS EDM )\]


file Name: \{datasetID\}-{filename}

I.e. 



<details>
<summary> <b>ev2a</b>  </summary>


- 01-curated

    - 00-allformats

        - {datasetID} 

            ...
    
    - 01-dimple

        - 01-pdb
        
            - {datasetID} 

                ...
            
        - 02-edm

            - {datasetID} 

            ...

    - 02-pandda

        - 01-pdb
        
            - 01-input

            - 02-model

        - 02-mtz

            - 01-meanmap

            - 02-zmap

            - 03-events

    - 03-refine

        - 01-pdb

            - 01-ensemble ()

            - 02-overlay (refine)

            - 03-ground (split.ground)

            - 04-bound
        
        - 02-edm (refine.mtz or ccp4)

  
    - 04-depo


    - 05-ligand

        - 
    
- 02-automated

</details>

In [1]:
def even(x):
    if x % 2 == 0:
        return x

list( filter( even, [1,2,3,4,5]))

[2, 4]

In [2]:
from pathlib import Path
rootPath = """../../../data/ev2a"""
rootPath = Path( rootPath )
iefolder = rootPath /"01-curated/00-allformats/A71EV2A-x0194"
iefolder = iefolder.resolve()

print( Path(rootPath).resolve())
print( iefolder)


/dls/science/groups/i04-1/software/xchem-data-enjoyers/alex-belo/xaidar/data/ev2a
/dls/science/groups/i04-1/software/xchem-data-enjoyers/alex-belo/xaidar/data/ev2a/01-curated/00-allformats/A71EV2A-x0194


In [3]:
# Filtering files in a folder with regex
print( [ path.name for path in iefolder.glob( "*event*_map*ccp4")  ] )
import re
from pathlib import Path

def glob_re( pattern, strings ):
    return filter( re.compile( pattern).search, strings)

print(  list( glob_re( ".*event.*_map.*ccp4$",  [ path.name for path in iefolder.iterdir() ] ) ) )



def glob_re( pattern, paths: list[Path] , outputPath = True):  
    if outputPath:
        def pathFilter( path: Path ):
            return re.compile( pattern).search( path.name)
    else:
        paths = [ path.name for path in paths]
        def pathFilter( path: Path ):
            return re.compile( pattern).search( path)
    return filter( pathFilter, paths)
print( list( glob_re( "event.*\.ccp4$", iefolder.iterdir(), outputPath = True  ) ) )
print( list( glob_re( "event.*\.ccp4$", iefolder.iterdir(), outputPath = False  ) ) )

['A71EV2A-x0194-event_1_1-BDC_0.35_map.native.ccp4', 'A71EV2A-x0194-event_1_1-BDC_0.67_map.native.ccp4', 'A71EV2A-x0194-event_2_1-BDC_0.69_map.native.ccp4']
['A71EV2A-x0194-event_1_1-BDC_0.35_map.native.ccp4', 'A71EV2A-x0194-event_1_1-BDC_0.67_map.native.ccp4', 'A71EV2A-x0194-event_2_1-BDC_0.69_map.native.ccp4']
[PosixPath('/dls/science/groups/i04-1/software/xchem-data-enjoyers/alex-belo/xaidar/data/ev2a/01-curated/00-allformats/A71EV2A-x0194/A71EV2A-x0194-event_1_1-BDC_0.35_map.native.ccp4'), PosixPath('/dls/science/groups/i04-1/software/xchem-data-enjoyers/alex-belo/xaidar/data/ev2a/01-curated/00-allformats/A71EV2A-x0194/A71EV2A-x0194-event_1_1-BDC_0.67_map.native.ccp4'), PosixPath('/dls/science/groups/i04-1/software/xchem-data-enjoyers/alex-belo/xaidar/data/ev2a/01-curated/00-allformats/A71EV2A-x0194/A71EV2A-x0194-event_2_1-BDC_0.69_map.native.ccp4')]
['A71EV2A-x0194-event_1_1-BDC_0.35_map.native.ccp4', 'A71EV2A-x0194-event_1_1-BDC_0.67_map.native.ccp4', 'A71EV2A-x0194-event_2_1-BDC

In [ ]:
from pathlib import Path
from shutil import copy2
rootPath = """../../../data/ev2a"""
rootPath = Path( rootPath )
iefolder = rootPath /"01-curated/00-allformats/A71EV2A-x0194"

saveDir = rootPath / "00-test"
saveDir.mkdir( parents=True, exist_ok = True)

datasetNumber = iefolder.name[8:]
molType = "all"

fileExtractFormat={"dimple":["dimple\.pdb$", "dimple\.mtz$"],

                "pandda_input":[".*pandda-input\.pdb$", ".*pandda-input\.mtz$"],
                "pandda_mean_map":[".*ground-state-average-map.*\.ccp4$"],
                "pandda_z_map": [".*z_map.*\.ccp4$"],
                "pandda_event_map": [".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.ccp4$", 
                              ".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.mtz$"],
                "pandda_model": [".*pandda-model\.pdb$"],

                "refine_ensemble":[".*ensemble-model\.pdb$"],
                "refine_merge":["refine\.pdb$","refine\.mtz$", "refine\.ccp4$"],
                "refine_ground": ["refine\.split\.ground-state\.pdb$"],
                "refine_bound": ["refine\.split\.bound-state\.pdb$"],

                "depo": ["[^s][^f]\.mmcif$", "sf.mmcif$"]}


for fileLabel, patterns in fileExtractFormat.items():
    for pattern in patterns:
        try:
            lst_paths = list(glob_re(pattern, iefolder.iterdir(), outputPath=True))
            lst_files = list(glob_re(pattern, iefolder.iterdir(), outputPath=False))
        except:
            print( f"Could not find matches for {pattern}")
        fileType="."+pattern.split(".")[-1][:-1] if pattern != "sf.mmcif$" else "_sf.mmcif"
        # sf mmcifs sometimes are found as .sf_mmcif or sf.mmcif

        # if fileLabel == "event_map":
        for idx, (file, path) in enumerate( zip(lst_files, lst_paths) ):
            if fileLabel == "pandda_event_map": # only label w many subtypes
                new_file_name= "{}-{}-pandda_{}_{}{}".format(dataset, molType,
                    re.search( "event_[0-9]*_[0-9]*", file).group(),
                    re.search( "BDC_[0-9|.]*", file).group(), fileType)
            else:
                if idx == 0:
                    new_file_name = "{}-{}-{}{}".format(dataset, molType,
                                                        fileLabel, fileType )
                else: # if more than one file ided for a specific regex
                    new_file_name = "{}-{}-{}_{}{}".format(dataset, molType,
                                                        idx, fileLabel, fileType )
                
            new_file_path = saveDir / new_file_name
            copy2( path.as_posix(), new_file_path.as_posix() )


In [ ]:
string = ("pandda_event_map_{}_{}_BDC_{}.ccp4".format(\""
                "re.search( \"event_[0-9]*\""
                ", file).group()))

print(string)


pandda_event_map_{"re.search( "event_[0-9]*", file).group()}_{}_BDC_{}.ccp4


In [7]:
from pathlib import Path
import re
filePath = Path( "data/ev2a/01-curated/00-allformats/A71EV2A-x0194/A71EV2A-x0194-event_2_1-BDC_0.69_map.native.ccp4" )
fileName = filePath.name

def foo( file):
    fileName ="pandda_event_map_{}_{}.ccp4".format(
                re.search( "event_[0-9]*_[0-9]*", file).group(),
                re.search( "BDC_[0-9|.]*", file).group()
         )
    return fileName

test = {".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.ccp4$" : foo }
file = fileName

test[".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.ccp4$"](file)

'pandda_event_map_event_2_1_BDC_0.69.ccp4'

In [ ]:
def events_name(origFileName):
        newFileName="pandda_event_map_{}_{}.ccp4".format(
                re.search( "event_[0-9]*_[0-9]*", origFileName).group(),
                re.search( "BDC_[0-9|.]*", origFileName).group())
        return newFileName
     
    

namingConvention={ # "Name" : "Regex"
                "dimple.pdb":"dimple\.pdb$", 
                "dimple.mtz": "dimple\.mtz$",

                "pandda_input.pdb": ".*pandda-input\.pdb$",
                "pandda_input.mtz":".*pandda-input\.mtz$",
                "pandda_input.ccp4": ".*pandda-input\.ccp4$",
                "pandda_mean_map.ccp4":".*ground-state-average-map.*\.ccp4$",
                "pandda_mean_map.mtz":".*ground-state-average-map.*\.mtz$",
                "pandda_z_map.ccp4":".*z_map.*\.ccp4$",
                "pandda_event_map_{}_{}_BDC_{}.ccp4":
                        ".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.ccp4$", 
                "pandda_event_map_{}_{}_BDC_{}.mtz":
                        ".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.mtz$",
                "pandda_model.pdb":".*pandda-model\.pdb$",

                "refine_ensemble.pdb":".*ensemble-model\.pdb$",
                "refine_merge.pdb":"refine\.pdb$",
                "refine_merge.mtz":"refine\.mtz$", 
                "refine_merge.ccp4":"refine\.ccp4$",
                "refine_ground.pdb":"refine\.split\.ground-state\.pdb$",
                "refine_bound.pdb":"refine\.split\.bound-state\.pdb$",

                "depo.mmcif":"[^s][^f]\.mmcif$",
                "depo_sf.mmcif":"sf.mmcif$"
                }

In [ ]:
from shutil import copy2


def cp_rename_files(fileExtractFormat, sourceFolder, saveFolder, molType = "all", 
                    dataset = "x-0194" ):
    for fileLabel, patterns in fileExtractFormat.items():
        for pattern in patterns:
            try:
                lst_paths = list(glob_re(pattern, sourceFolder.iterdir(), outputPath=True))
                lst_files = list(glob_re(pattern, sourceFolder.iterdir(), outputPath=False))
            except:
                print( f"Could not find matches for {pattern}")
            fileType="."+pattern.split(".")[-1][:-1] if pattern != "sf.mmcif$" else "_sf.mmcif"
            # sf mmcifs sometimes are found as .sf_mmcif or sf.mmcif

            # if fileLabel == "event_map":
            for idx, (file, path) in enumerate( zip(lst_files, lst_paths) ):
                if fileLabel == "pandda_event_map": # only label w many subtypes
                    new_file_name= "{}-{}-pandda_{}_{}{}".format(dataset, molType,
                        re.search( "event_[0-9]*_[0-9]*", file).group(),
                        re.search( "BDC_[0-9|.]*", file).group(), fileType)
                else:
                    if idx == 0:
                        new_file_name = "{}-{}-{}{}".format(dataset, molType,
                                                            fileLabel, fileType )
                    else: # if more than one file ided for a specific regex
                        new_file_name = "{}-{}-{}_{}{}".format(dataset, molType,
                                                            idx, fileLabel, fileType )
                    
                new_file_path = saveFolder / new_file_name
                copy2( path.as_posix(), new_file_path.as_posix() )
    return None

In [8]:
from pathlib import Path

rootPath = """../../../data/ev2a"""
rootPath = Path( rootPath )
iefolder = rootPath /"01-curated/00-allformats/A71EV2A-x0194"

saveDir = rootPath / "00-test"
saveDir.mkdir( parents=True, exist_ok = True)

datasetNumber = iefolder.name[8:]
molType = "all"

fileExtractFormat={"dimple":["dimple\.pdb$", "dimple\.mtz$"],

                "pandda_input":[".*pandda-input\.pdb$", ".*pandda-input\.mtz$"],
                "pandda_mean_map":[".*ground-state-average-map.*\.ccp4$"],
                "pandda_z_map": [".*z_map.*\.ccp4$"],
                "pandda_event_map": [".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.ccp4$", 
                              ".*event_[0-9]*_[0-9]*-BDC_[0-9|.]*_map.*\.mtz$"],
                "pandda_model": [".*pandda-model\.pdb$"],

                "refine_ensemble":[".*ensemble-model\.pdb$"],
                "refine_merge":["refine\.pdb$","refine\.mtz$", "refine\.ccp4$"],
                "refine_ground": ["refine\.split\.ground-state\.pdb$"],
                "refine_bound": ["refine\.split\.bound-state\.pdb$"],

                "depo": ["[^s][^f]\.mmcif$", "sf.mmcif$"]}

for sourceFolder in rootPath.joinpath("01-curated/00-allformats").iterdir():
    datasetNumber = sourceFolder.name[8:]
    cp_rename_files( fileExtractFormat, sourceFolder, saveDir, molType = "all", 
                    dataset = datasetNumber)

In [18]:
from pathlib import Path

rootPath = """../../../data/ev2a"""
rootPath = Path( rootPath )

dataDir = rootPath / "00-test"


lst = list( glob_re( ".*pandda_model.*pdb$", list( dataDir.iterdir() ) , outputPath = True ) )
pymolPath = "../../../../PyMOL/pandda-models.txt"
with open( pymolPath, "w") as file:
    file.writelines( [ path.resolve().as_posix()+"   "+path.name[:5]+"\n" for path in lst ])

lst = list( glob_re( "refine.*ground.*.*pdb$", list( dataDir.iterdir() ) , outputPath = True ) )
pymolPath = "../../../../PyMOL/refine-ground.txt"
with open( pymolPath, "w") as file:
    file.writelines( [ path.resolve().as_posix()+"   "+path.name[:5]+"\n" for path in lst ])

lst = list( glob_re( "refine.*bound.*.*pdb$", list( dataDir.iterdir() ) , outputPath = True ) )
pymolPath = "../../../../PyMOL/refine-bound.txt"
with open( pymolPath, "w") as file:
    file.writelines( [ path.resolve().as_posix()+"   "+path.name[:5]+"\n" for path in lst ])

---

In [ ]:
fileExtractFormat = {"dimple.pdb":["dimple.pdb"],
                     "dimple.mtz":["dimple.mtz"],
                                      
        
                    "pandda_input.pdb":["*pandda-input.pdb"],
                    "pandda_input.mtz":["*pandda-input.mtz"],
                    "mean_map.ccp4":["*ground-state-averaga-map*.ccp4"],
                    "z_map.ccp4":["*z_map*.ccp4"]
                    "event_.ccp4":["*event_"]

                                "02-model":["*pandda-model.pdb"],
                                },
                            "02-edm":{
                                "01-meanmap":["*average-map.native.ccp4"],
                                "02-zmap":["*z_map.native.ccp4"],
                                "03-events":["*-event-*.ccp4", "*-event-*.mtz"],
                                },
                        },
                        "03-refine":{
                            "01-pdb":{
                                "01-ensemble":["*ensemble-model.pdb"],
                                "02-overlay": ["refine.pdb"],
                                "03-ground":["refine.split.ground-state.pdb"],
                                "04-bound":["refine.split.bound-state.pdb"],
                                },
                            "02-edm":["refine.ccp4", "refine.mtz"],
                        },
                        "04-depo":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "05-ligand":{
                            "01-pdb":[],
                            "02-edm":[],
                        },

                    },                        
                    "02-automated":[],                   
                }

In [ ]:
regexes = {"dimple": "*dimple*", 
           
           "pandda_input": "*pandda-input*", 
           "mean_map":"*ground-state-average-map*", 
           "z_map": "*z_map*", "pandda_model": "*pandda-model*",
           
           "refine_ensemble": "*ensemble-model*",
             "refine": "*refine*"  }

In [ ]:
for dataset in rootPath.joinpath( "01-curated/00-allformats"):
    datasetName = dataset.name[8:]
    for fileProcessType, regex in regexes: 
        for file in dataset.glob(regex):
            fileType = file.name.split(".")[-1]
            

    # for file in dataset.iterdir()


In [12]:
import re
test = "sjfladks.mmcif"

re.search( ".*[^s][^f]\.mmcif$", test)
# re.search("", test)

<re.Match object; span=(0, 14), match='sjfladks.mmcif'>

In [3]:
lst1 = [""]
lst2 = ["a", "b"]
lst3 = [ number + letter for number in lst1 
        for letter in lst2]
print( lst3)

['a', 'b']


In [ ]:
def buildRegexList( current_patterns, lst_patterns, regex = True):
    if isinstance( lst_patterns, list)  :
        updated_patterns = [ regex + pattern + "-" for regex in current_patterns 
                     for pattern in lst_patterns]
    else:
        if regex:
            updated_patterns = [ regex + ".*" + "-" for regex in current_patterns]
        else:
            updated_patterns = [ regex + "*" + "-" for regex in current_patterns]
    return updated_patterns



def filterPaths( datasets: None | list = [""], molTypes : None | list = None,
                 processSubTypes : None | list  = None, fileTypes: None | list  = None,
                 regex: bool = True):
    lst_regex = [ "^" ] if regex else [""]
    for fileNameElement in [datasets, molTypes, processSubTypes, fileTypes]:
        lst_regex = buildRegexList(lst_regex, fileNameElement, regex = regex )
    lst_regex = [ element[:-1] for element in lst_regex]

    # if isinstance( datasets, list)  :
    #     lst_regex = [ regex + dataset +  for regex in lst_regex 
    #                  for dataset in datasets]
    # if isinstance( molTypes, list):
    #     lst_regex = [ regex + molType for regex in lst_regex 
    #                  for molType in molTypes]
    return lst_regex


[ [], ["all"], ["refine_ground"], []]

def translateKeysToRegex( key ):
    processTranslation = { "dimple": "dimple",
                          "pandda_input": "pandda_input", 
                            "mean_map": ""}

    translation = { "datasets": key[0]}
    translation = { "molTypes": key[1] }
    translation = 

In [18]:
filterPaths( datasets = ["x0140"], molTypes = ["all"], 
            processSubTypes= ["refine_ground","refine_bound"], 
            fileTypes = [".pdb"], regex = False)

['x0140-all-refine_ground-.pdb', 'x0140-all-refine_bound-.pdb']

In [ ]:

filesTypesDict = {
    
    # Data File Type
    "pdb": [".*\.pdb$"],
    "mmcif":[".*\.mmcif$"],
    "ccp4": [".*\.ccp4$"],
    "mtz": [".*\.mtz$"],

    # Processing Subtypes
    "dimple":[],

    "pandda-input": [],
    "z-map": [],
    "mean-map": [],
    "event-map":[],
    "pandda-model":[],



    # Data Model Type
    "molec": [".*\.pdb$", ".*[^s][^f]\.mmcif$", ],
    "edm": [ ".*\.ccp4$", ".*\.mtz$", ".*sf\.mmcif$"],
    
    # Processing Type
    "dimple":[],
    "pandda":[],
    "refine":[],
    "deposit":[],

    # Bind Type
    "ground":[],
    # "apo":[],
    "bound":[".*bound-state.*", ".*pandda-model.*", ""],
    # "holo":[],

    # Molec Type
    "ligand":[],    # ------> Needs completing
    "solvent":[],
    "protein":[],
    "water":[],



}
import re

def buildRegexList( current_patterns, lst_patterns, regex = True):
    if isinstance( lst_patterns, list)  :
        updated_patterns = [ regex + pattern + "-" for regex in current_patterns 
                     for pattern in lst_patterns]
    else:
        if regex:
            updated_patterns = [ regex + ".*" + "-" for regex in current_patterns]
        else:
            updated_patterns = [ regex + "*" + "-" for regex in current_patterns]
    return updated_patterns



def filterPaths( datasets: None | list = [""], molTypes : None | list = None,
                 processSubTypes : None | list  = None, fileTypes: None | list  = None,
                 regex: bool = True):
    lst_regex = [ "^" ] if regex else [""]
    for fileNameElement in [datasets, molTypes, processSubTypes, fileTypes]:
        lst_regex = buildRegexList(lst_regex, fileNameElement )
    

    # if isinstance( datasets, list)  :
    #     lst_regex = [ regex + dataset +  for regex in lst_regex 
    #                  for dataset in datasets]
    # if isinstance( molTypes, list):
    #     lst_regex = [ regex + molType for regex in lst_regex 
    #                  for molType in molTypes]
    return lst_regex

def filterFiles( lstPaths: list, filters: str | list ) -> list:
    filteredLstPaths = lstPaths
    if isinstance( filters, str):
        filters = [filters]
    lst_regexs = [ regex for regex in filesTypesDict[filter] 
                  for filter in filters ]
    for regex in lst_regexs:
        regex = re.compile( regex)
        filteredLstPaths = [ path for path in filteredLstPaths 
                            if regex.search( path) ]
    


    return filteredLstPaths

In [1]:
def foo( arg1, arg2):

    return arg1 + arg2
foo( foo(1,1), 1)

3

---


In [11]:
directoryFormat = { "01-curated": {
                        "00-allformats": [],
                        "01-dimple":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "02-pandda":{
                            "01-pdb":{
                                "01-input":[],
                                "02-model":[],
                                },
                            "02-edm":{
                                "01-meanmap":[],
                                "02-zmap":[],
                                "03-events":[],
                                },
                        },
                        "03-refine":{
                            "01-pdb":{
                                "01-ensemble":[],
                                "02-overlay":[],
                                "03-ground":[],
                                "04-bound":[],
                                },
                            "02-edm":[],
                        },
                        "04-depo":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "05-ligand":{
                            "01-pdb":[],
                            "02-edm":[],
                        },

                    },                        
                    "02-automated":[],                   
                }

In [ ]:
from pathlib import Path

rootPath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a").mkdir( )

In [ ]:
def createWorkDir(rootDir: Path, dirFormat):
    for key, value in dirFormat.items():
        newPath = rootDir.joinpath(key)
        newPath.mkdir(parents=True, exist_ok=True)
        if isinstance(value, dict):
            createWorkDir(newPath, value)

In [ ]:
createWorkDir( rootPath, directoryFormat)

In [30]:
fileExtractFormat = { "01-curated": {
                        "00-allformats": [],
                        "01-dimple":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "02-pandda":{
                            "01-pdb":{
                                "01-input":["*pandda-input.pdb"],
                                "02-model":["*pandda-model.pdb"],
                                },
                            "02-edm":{
                                "01-meanmap":["*average-map.native.ccp4"],
                                "02-zmap":["*z_map.native.ccp4"],
                                "03-events":["*-event-*.ccp4", "*-event-*.mtz"],
                                },
                        },
                        "03-refine":{
                            "01-pdb":{
                                "01-ensemble":["*ensemble-model.pdb"],
                                "02-overlay": ["refine.pdb"],
                                "03-ground":["refine.split.ground-state.pdb"],
                                "04-bound":["refine.split.bound-state.pdb"],
                                },
                            "02-edm":["refine.ccp4", "refine.mtz"],
                        },
                        "04-depo":{
                            "01-pdb":[],
                            "02-edm":[],
                        },
                        "05-ligand":{
                            "01-pdb":[],
                            "02-edm":[],
                        },

                    },                        
                    "02-automated":[],                   
                }

In [19]:
print( list(rootPath.glob("*")))

[PosixPath('/home/eoo22534/MyDB/xaidar/data/ev2a/01-curated'), PosixPath('/home/eoo22534/MyDB/xaidar/data/ev2a/02-automated')]


In [28]:
import re
test = "A71EV2A-x0194-event_1_1-BDC_0.35_map.native.mtz"
test2 = "dimple.pdb"
dataset = "A71EV2A-x0194"[8:]

for file in [test, test2]:
    if  re.search( dataset, file):
        print( re.search( dataset, file).end() )
        print( file[re.search( dataset, file).end()+1:])


13
event_1_1-BDC_0.35_map.native.mtz


In [36]:
sourcePath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a/01-curated/00-allformats")
rootPath = Path("/home/eoo22534/MyDB/xaidar/data/ev2a")
from shutil import copy2
import re

def cpFiles(sourcePath, targetPath, targetFiles):
    for patrnMatch in targetFiles:
        for dir in sourcePath.glob("*"):
            dataset = dir.name[8:]
            for file in dir.glob( patrnMatch):

                if  re.search( dataset, file.name):
                    fileName = file.name[re.search( dataset, file.name).end()+1:]
                else:
                    fileName = file.name

                newPath = targetPath.joinpath( f"{dataset}-{fileName}")
                copy2( file.as_posix(), newPath.as_posix())


def movFiles(rootDir, filesToExtract, sourcePath):
    for key, value in filesToExtract.items():
        newPath = rootDir.joinpath( key)

        if isinstance( value, dict):
            movFiles( newPath, filesToExtract[key], sourcePath)
        elif isinstance( value, list) and value != []:
            cpFiles( sourcePath, newPath, value)
            
movFiles(rootPath, fileExtractFormat,  sourcePath)